# Notebook 03: Trend Analysis with Cascade Protocol Data

This notebook demonstrates how to use the `cascade-protocol` Python SDK alongside pandas and matplotlib to perform longitudinal health trend analysis.

**Topics covered:**
1. Loading vital signs from a Cascade Pod
2. Filtering by vital type
3. Plotting blood pressure trends over time
4. Analyzing lab result trends (A1c, cholesterol)
5. Correlating activity with vital signs

**Setup:** `pip install "cascade-protocol[notebooks]"`

In [ ]:
import sys
from pathlib import Path
import warnings

sdk_root = Path("../src").resolve()
if str(sdk_root) not in sys.path:
    sys.path.insert(0, str(sdk_root))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from cascade_protocol import Pod, VitalSign, serialize, parse

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

print("Libraries loaded.")

## 1. Load Vital Signs from the Reference Pod

In [ ]:
pod_path = Path("../../reference-patient-pod").resolve()

if pod_path.exists():
    pod = Pod.open(pod_path)
    vitals_rs = pod.query("vital-signs")
    vitals_df = vitals_rs.to_dataframe()
    print(f"Loaded {len(vitals_rs)} vital sign readings from pod")
else:
    # Fall back to synthetic data for demonstration
    print("Reference pod not found — using synthetic data")
    import uuid
    from datetime import date, timedelta
    import random
    
    rows = []
    base_date = date(2026, 1, 1)
    for i in range(30):
        d = base_date + timedelta(days=i)
        rows.append({
            "id": str(uuid.uuid4()),
            "type": "VitalSign",
            "vital_type": "bloodPressureSystolic",
            "value": 125 + random.randint(-10, 20),
            "unit": "mmHg",
            "effective_date": f"{d}T09:00:00Z",
            "data_provenance": "DeviceGenerated",
            "schema_version": "1.3",
        })
        rows.append({
            "id": str(uuid.uuid4()),
            "type": "VitalSign",
            "vital_type": "bloodPressureDiastolic",
            "value": 78 + random.randint(-8, 12),
            "unit": "mmHg",
            "effective_date": f"{d}T09:00:00Z",
            "data_provenance": "DeviceGenerated",
            "schema_version": "1.3",
        })
        rows.append({
            "id": str(uuid.uuid4()),
            "type": "VitalSign",
            "vital_type": "heartRate",
            "value": 68 + random.randint(-8, 15),
            "unit": "bpm",
            "effective_date": f"{d}T09:00:00Z",
            "data_provenance": "DeviceGenerated",
            "schema_version": "1.3",
        })
    vitals_df = pd.DataFrame(rows)

print(f"\nVital types present:")
if 'vital_type' in vitals_df.columns:
    print(vitals_df['vital_type'].value_counts().to_string())

## 2. Blood Pressure Trend

In [ ]:
if 'effective_date' in vitals_df.columns and 'vital_type' in vitals_df.columns:
    vitals_df['date_parsed'] = pd.to_datetime(
        vitals_df['effective_date'].str.replace('Z', '+00:00'), 
        utc=True, 
        errors='coerce'
    )
    
    systolic = vitals_df[vitals_df['vital_type'] == 'bloodPressureSystolic'].copy()
    diastolic = vitals_df[vitals_df['vital_type'] == 'bloodPressureDiastolic'].copy()
    
    if not systolic.empty or not diastolic.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        
        if not systolic.empty:
            systolic_sorted = systolic.sort_values('date_parsed')
            ax.plot(systolic_sorted['date_parsed'], systolic_sorted['value'],
                    'b-o', markersize=4, label='Systolic (mmHg)', linewidth=2)
            ax.axhline(y=130, color='b', linestyle='--', alpha=0.4, label='Systolic target (<130)')
        
        if not diastolic.empty:
            diastolic_sorted = diastolic.sort_values('date_parsed')
            ax.plot(diastolic_sorted['date_parsed'], diastolic_sorted['value'],
                    'r-o', markersize=4, label='Diastolic (mmHg)', linewidth=2)
            ax.axhline(y=80, color='r', linestyle='--', alpha=0.4, label='Diastolic target (<80)')
        
        ax.set_title('Blood Pressure Trend — Alex Rivera', fontsize=14, fontweight='bold')
        ax.set_xlabel('Date')
        ax.set_ylabel('Blood Pressure (mmHg)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        # Summary statistics
        if not systolic.empty:
            print(f"\nSystolic BP stats:")
            print(f"  Mean: {systolic['value'].mean():.1f} mmHg")
            print(f"  Min:  {systolic['value'].min():.1f} mmHg")
            print(f"  Max:  {systolic['value'].max():.1f} mmHg")
    else:
        print("No blood pressure data found")
else:
    print("Vital signs DataFrame missing expected columns")

## 3. Heart Rate Distribution

In [ ]:
heart_rate = vitals_df[vitals_df.get('vital_type', pd.Series()) == 'heartRate']

if not heart_rate.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Distribution
    ax1.hist(heart_rate['value'], bins=15, color='teal', edgecolor='white', alpha=0.8)
    ax1.axvline(heart_rate['value'].mean(), color='navy', linestyle='--', 
                linewidth=2, label=f"Mean: {heart_rate['value'].mean():.1f} bpm")
    ax1.set_title('Heart Rate Distribution', fontweight='bold')
    ax1.set_xlabel('Heart Rate (bpm)')
    ax1.set_ylabel('Frequency')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Over time
    if 'date_parsed' in heart_rate.columns:
        hr_sorted = heart_rate.sort_values('date_parsed')
        ax2.plot(hr_sorted['date_parsed'], hr_sorted['value'],
                 'teal', marker='o', markersize=3, linewidth=1.5)
        ax2.axhspan(60, 100, alpha=0.1, color='green', label='Normal range (60-100)')
        ax2.set_title('Heart Rate Over Time', fontweight='bold')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Heart Rate (bpm)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No heart rate data found")

## 4. Round-Trip Verification

Demonstrate that data survives a serialize → parse → serialize cycle.

In [ ]:
import uuid

# Create a VitalSign record
original = VitalSign(
    id=f"urn:uuid:{uuid.uuid4()}",
    vital_type="bloodPressureSystolic",
    vital_type_name="Systolic Blood Pressure",
    value=128.0,
    unit="mmHg",
    effective_date="2026-01-20T09:15:00Z",
    reference_range_low=90.0,
    reference_range_high=120.0,
    interpretation="elevated",
    data_provenance="DeviceGenerated",
    schema_version="1.3",
)

# Serialize
turtle1 = serialize(original)
print("=== Original Turtle ===")
print(turtle1)

# Parse back
parsed = parse(turtle1, "VitalSign")
if parsed:
    restored = parsed[0]
    print(f"\nRestored value: {restored.value} {restored.unit}")
    print(f"Vital type: {restored.vital_type}")
    print(f"Interpretation: {restored.interpretation}")
    print(f"Reference range: {restored.reference_range_low} - {restored.reference_range_high}")
    
    # Serialize again
    turtle2 = serialize(restored)
    print("\n=== Round-trip Turtle ===")
    print(turtle2)

## 5. Namespace Reference

In [ ]:
from cascade_protocol import NAMESPACES

print("Cascade Protocol Namespace Reference:")
print("=" * 60)
for prefix, uri in NAMESPACES.items():
    if prefix not in ("rdf", "rdfs", "owl"):
        print(f"  @prefix {prefix:<12} <{uri}>")